In [1]:
import mlflow

mlflow.set_experiment("house_pipeline")

2026/04/09 18:10:13 INFO mlflow.tracking.fluent: Experiment with name 'house_pipeline' does not exist. Creating a new experiment.


<Experiment: artifact_location='file:d:/hd/mlruns/3', creation_time=1775751013565, experiment_id='3', last_update_time=1775751013565, lifecycle_stage='active', name='house_pipeline', tags={}, trace_location=None, workspace='default'>

In [2]:
import pandas as pd

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor

from sklearn.datasets import fetch_california_housing
# import pandas as pd

data = fetch_california_housing()

X = pd.DataFrame(
    data.data,
    columns=data.feature_names
)

y = data.target

pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("model", RandomForestRegressor())
])

In [4]:
X.head()

,MedInc,HouseAge,AveRooms,AveBedrms,Population,AveOccup,Latitude,Longitude
0,8.3252,41.0,6.984127,1.023810,322.0,2.555556,37.88,-122.23
1,8.3014,21.0,6.238137,0.971880,2401.0,2.109842,37.86,-122.22
2,7.2574,52.0,8.288136,1.073446,496.0,2.802260,37.85,-122.24
3,5.6431,52.0,5.817352,1.073059,558.0,2.547945,37.85,-122.25
4,3.8462,52.0,6.281853,1.081081,565.0,2.181467,37.85,-122.25


In [6]:
y

array([4.526, 3.585, 3.521, ..., 0.923, 0.847, 0.894], shape=(20640,))

In [8]:
from sklearn.model_selection import train_test_split, GridSearchCV

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [9]:
param_grid = {
    "model__n_estimators": [50, 100],
    "model__max_depth": [None, 5]
}

In [11]:
from sklearn.metrics import mean_squared_error
from itertools import product

mlflow.sklearn.autolog()

keys = param_grid.keys()
values = param_grid.values()

for combination in product(*values):

    params = dict(zip(keys, combination))

    with mlflow.start_run():

        pipeline.set_params(**params)

        pipeline.fit(
            X_train,
            y_train
        )

        y_pred = pipeline.predict(
            X_test
        )

        mse = mean_squared_error(
            y_test,
            y_pred
        )

        mlflow.log_params(params)

        mlflow.log_metric(
            "mse",
            mse
        )

2026/04/09 18:25:35 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/04/09 18:25:41 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/04/09 18:25:52 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mec

In [ ]:
with mlflow.start_run(run_name="RF Hyperparameter Tuning"):
    grid_search = GridSearchCV(pipeline, param_grid, cv=5, scoring="neg_mean_squared_error", n_jobs=-1)
    grid_search.fit(X_train, y_train)

    best_score = grid_search.score(X_test, y_test)
    print(f"Best params: {grid_search.best_params_}")
    print(f"Best CV score: {grid_search.best_score_:.3f}")
    print(f"Test score: {best_score:.3f}")

2026/04/09 18:51:33 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/04/09 18:51:36 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/04/09 18:51:39 INFO mlflow.sklearn.utils: Logging the 5 best runs, no runs will be omitted.


Best params: {'model__max_depth': None, 'model__n_estimators': 100}
Best CV score: -0.262
Test score: -0.255


In [ ]:
best_mse = 1
best_pipeline = None

with mlflow.start_run(run_name="random_forest"):

    for combination in product(*values):

        params = dict(zip(keys, combination))

        with mlflow.start_run(nested=True):

            pipeline.set_params(
                **params
            )

            pipeline.fit(
                X_train,
                y_train
            )

            y_pred = pipeline.predict(
                X_test
            )

            mse = mean_squared_error(
            y_test,
            y_pred
            )

            mlflow.log_params(params)

            mlflow.log_metric(
                "mse",
                mse
            )

            if mse < best_mse:
                best_mse = mse
                best_pipeline = pipeline

2026/04/09 18:41:44 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/04/09 18:41:49 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/04/09 18:42:00 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mec

In [ ]:
import mlflow.sklearn
from mlflow.models import infer_signature

pipeline = best_pipeline

pipeline.fit(
    X_train,
    y_train
)

signature = infer_signature(
    X_train,
    pipeline.predict(X_train)
)

with mlflow.start_run():

    mlflow.sklearn.log_model(
        pipeline,
        "house_model",
        signature=signature
    )

2026/04/09 19:09:06 INFO mlflow.utils.autologging_utils: Created MLflow autologging run with ID 'e3ed447fac174029baee83a347628ae0', which will track hyperparameters, performance metrics, model artifacts, and lineage information for the current sklearn workflow
2026/04/09 19:09:09 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/04/09 19:09:12 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/09 19:09:12 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitra